In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## Upload Lora Adapters

In [ ]:
from google.colab import files
files.upload()

Saving intent_classification_v1.zip to intent_classification_v1.zip


In [ ]:
!unzip intent_classification_v1.zip -d intent_classification_v1

Archive:  intent_classification_v1.zip
   creating: intent_classification_v1/content/intent_classification_v1/
  inflating: intent_classification_v1/content/intent_classification_v1/special_tokens_map.json  
  inflating: intent_classification_v1/content/intent_classification_v1/README.md  
  inflating: intent_classification_v1/content/intent_classification_v1/adapter_config.json  
  inflating: intent_classification_v1/content/intent_classification_v1/tokenizer_config.json  
  inflating: intent_classification_v1/content/intent_classification_v1/adapter_model.safetensors  
  inflating: intent_classification_v1/content/intent_classification_v1/tokenizer.json  


## Inference Class

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seg_length = 2048
dtype = None
load_in_4bit = True

class IntentClassification:
    def __init__(self, model_path):
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name = model_path,
            max_seq_length = max_seg_length,
            dtype = dtype,
            load_in_4bit = load_in_4bit
        )
        FastLanguageModel.for_inference(self.model)

    def __call__(self, message):
        prompt = f"""### Instruction:
Classify the intent of the following banking query. Return only the intent label.

### Input:
{message}

### Response:
"""
        inputs = self.tokenizer([prompt], return_tensors = "pt").to("cuda")
        outputs = self.model.generate(**inputs, max_new_tokens = 128, use_cache = True)
        decoded_output = self.tokenizer.batch_decode(outputs)[0]

        predicted_label = decoded_output.split("### Response:")[-1].strip()
        predicted_label = predicted_label.split("\n")[0]
        return predicted_label

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
classifier = IntentClassification("intent_classification_v1/content/intent_classification_v1")

==((====))==  Unsloth 2026.4.5: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

Unsloth 2026.4.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


## Predict 1 input

In [ ]:
print(classifier("Why does my statement have these extra charges?"))

extra_charge_on_statement<|end_of_text|>


# Inference on the test dataset

## Download test dataset

In [ ]:
from datasets import load_dataset
dataset = load_dataset("banking77")
test_ds = dataset["test"]

label_names = test_ds.features["label"].names

def add_label_text(example):
    example["label_text"] = label_names[example["label"]]
    return example

test_ds = test_ds.map(add_label_text)

test_sample = test_ds.shuffle(seed=42).select(range(200))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/298k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/93.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

## Save test dataset to csv file

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "text": test_sample["text"],
    "label": test_sample["label"],
    "label_text": test_sample["label_text"],
})

df.to_csv("test.csv", index=False)

print("Saved to test.csv")

Saved to test.csv


In [ ]:
def clean(text):
    return (
        text.lower()
            .replace("<|end_of_text|>", "")
            .replace("</s>", "")
            .strip()
            .split("\n")[0]
    )

## Compute Accuracy

In [ ]:
correct = 0
total = len(test_sample)

for i, example in enumerate(test_sample):
    text = example["text"]
    true = example["label_text"]

    pred = classifier(text)

    if clean(pred) == clean(true):
        correct += 1

    if i < 5:
        print("TEXT:", text)
        print("TRUE:", clean(true))
        print("PRED:", clean(pred))
        print("-" * 50)

accuracy = correct / total
print(f"\n Accuracy: {accuracy:.4f}")

TEXT: Are there restrictions for my disposable card since it does not seem to be working?
TRUE: virtual_card_not_working
PRED: disposable_card_limits
--------------------------------------------------
TEXT: I'm trying to transfer money to another country. It's just pending and not sending. My account details are correct. Please help me.
TRUE: pending_transfer
PRED: pending_transfer
--------------------------------------------------
TEXT: What do I do if I notice a strange withdrawl in my statement?
TRUE: cash_withdrawal_not_recognised
PRED: cash_withdrawal_not_recognised
--------------------------------------------------
TEXT: I got a message that I need to verify my identity; what do I do?
TRUE: verify_my_identity
PRED: verify_my_identity
--------------------------------------------------
TEXT: How do I use the app if I don't have my phone with me?
TRUE: lost_or_stolen_phone
PRED: lost_or_stolen_phone
--------------------------------------------------

 Accuracy: 0.9400
